---
Task 1 — Customer Support Triage Pipeline 

Goal: Build a multi-agent workflow that classifies incoming customer messages, retrieves relevant knowledge base (KB) content, and generates a customer-ready response. 

Steps: 
Create a Classifier Agent to identify customer intent (e.g., billing, technical issue, account access) and assign priority levels. 
Create a Retrieval Agent that searches the knowledge base and returns the top-K relevant articles or passages related to the identified intent. 
Create a Composer Agent that uses the customer query and retrieved KB content to draft a professional customer-facing response. 

Deliverables: 
Five sample customer queries. 

Agent execution trace showing:  
Classification result 
Retrieved KB passages 
Draft response generation 
Final customer-facing response for each sample query.


---

#### setup

In [6]:
import os
import json
import time
from typing import TypedDict, List
import anthropic
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("KEY")
base_url = os.getenv("BASE_URL")
model_name = os.getenv("MODEL")  # e.g., "claude-3-5-sonnet-20240620"

if not api_key or not model_name:
    raise ValueError("Missing 'KEY' or 'MODEL' environment variables! Please set them before running.")

# Initialize the Anthropic client (handles custom base URLs/proxies if provided)
if base_url:
    client = anthropic.Anthropic(api_key=api_key, base_url=base_url)
else:
    client = anthropic.Anthropic(api_key=api_key)

print(f"Anthropic client initialized successfully with model: {model_name}")

Anthropic client initialized successfully with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


#### Define State Schema and Mock Knowledge Base

In [7]:
# Define the pipeline data state structure
class SupportPipelineState(TypedDict):
    customer_query: str
    intent: str
    sub_intent: str
    priority: str
    retrieved_passages: List[dict]
    final_response: str

# In-memory documentation cache for intent matching
MOCK_KB = {
    "billing_dispute": "If a user is billed twice within the same billing cycle for a single subscription, verify the transaction IDs via Stripe. If both transactions are settled for the same billing period, initiate an immediate refund for the duplicate invoice. Refunds typically take 5-10 business days to clear back to the original payment method.",
    "technical_issue": "An isolated caching issue causes a crash on launch in version 4.2 for certain iOS devices. Workaround: Users must completely uninstall the application, navigate to iOS Settings > General > iPhone Storage to ensure all local data is purged, reboot the device, and perform a clean reinstall from the App Store. A permanent hotfix (v4.2.1) is currently in review.",
    "account_access": "For security and privacy compliance, support representatives cannot disable 2FA without absolute identity verification. Users locked out without backup codes must submit a secure identity verification request containing: 1) Account email, 2) The last 4 digits of the billing card on file, and 3) A government-issued photo ID match via our secure upload portal (link: security.platform.com/verify).",
    "subscription_management": "To cancel a premium subscription, users must click on their Profile Avatar (top-right) -> select 'Account Settings' -> navigate to the 'Billing & Plans' tab -> click the small gray text link 'Manage Subscription' under the plan details -> select 'Cancel Plan' and confirm.",
    "developer_api_issue": "The `/v1/analytics/export` endpoint has a hard synchronous gateway timeout limit of 30 seconds, which triggers a 504 error for payloads larger than 50MB. For large dataset extractions, developers must migrate to the asynchronous worker pool by calling `/v1/analytics/export/async` with a webhook callback payload parameter, then poll the status via `/v1/analytics/status/{job_id}`."
}

print("State graph structure and internal documentation registry completed.")

State graph structure and internal documentation registry completed.


#### Classification Node Implementation

In [8]:
def classifier_node(state: SupportPipelineState):
    """Uses Claude to classify customer intent into a strict JSON payload format."""
    prompt = f'Classify this customer ticket: "{state["customer_query"]}"'
    
    response = client.messages.create(
        model=model_name,
        max_tokens=256,
        temperature=0.0,
        system=(
            "You are a routing classification engine. Analyze the input ticket and respond ONLY with a raw JSON object. "
            "Do not include any conversational filler, intro, or markdown formatting.\n"
            "JSON structure: {\"intent\": \"str\", \"sub_intent\": \"str\", \"priority\": \"str\"}\n"
            "Allowed values for intent: \"billing_dispute\", \"technical_issue\", \"account_access\", \"subscription_management\", \"developer_api_issue\"\n"
            "Allowed values for priority: \"low\", \"medium\", \"high\", \"urgent\""
        ),
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw_content = response.content[0].text.strip()
    
    # Clean fallback processing in case markdown wrappers are returned
    if raw_content.startswith("```json"):
        raw_content = raw_content.split("```json")[1].split("```")[0].strip()
    elif raw_content.startswith("```"):
        raw_content = raw_content.split("```")[1].split("```")[0].strip()

    data = json.loads(raw_content)
    print(f"[AGENT_TRACE: CLASSIFIER] Intent Match -> {data['intent']} | Priority Level -> {data['priority'].upper()}")
    return {"intent": data["intent"], "sub_intent": data["sub_intent"], "priority": data["priority"]}

#### Retrieval and Composition Nodes

In [9]:
def retrieval_node(state: SupportPipelineState):
    """Router block looking up standard manuals based on target intent keys."""
    intent = state["intent"]
    kb_passage = MOCK_KB.get(intent, "Consult core policies for generic ecosystem concerns.")
    print(f"[AGENT_TRACE: RETRIEVAL] Extracted custom documentation matching workflow context.")
    return {"retrieved_passages": [{"article_id": "KB-ROUTED-CLAUDE", "passage": kb_passage}]}

def composer_node(state: SupportPipelineState):
    """Uses Claude to generate the final response by integrating context text frames with user variables."""
    kb_context = state["retrieved_passages"][0]["passage"]
    prompt = f"Customer Ticket: {state['customer_query']}\n\nKnowledge Base Guidelines to follow: {kb_context}"
    
    response = client.messages.create(
        model=model_name,
        max_tokens=1024,
        temperature=0.3,
        system="You are an expert customer support agent. Draft a clear, helpful, professional response to the user following the context rules provided exactly.",
        messages=[{"role": "user", "content": prompt}]
    )
    return {"final_response": response.content[0].text.strip()}

#### Compile the State Machine Graph

In [10]:
# Initialize graph workflow structure template
task1_graph = StateGraph(SupportPipelineState)

# Register defined execution functions into nodes
task1_graph.add_node("classifier", classifier_node)
task1_graph.add_node("retriever", retrieval_node)
task1_graph.add_node("composer", composer_node)

# Set up strict sequential processing directions
task1_graph.add_edge(START, "classifier")
task1_graph.add_edge("classifier", "retriever")
task1_graph.add_edge("retriever", "composer")
task1_graph.add_edge("composer", END)

# Finalize and compile runnable orchestration agent
triage_pipeline = task1_graph.compile()
print("LangGraph state machine successfully compiled and active.")

LangGraph state machine successfully compiled and active.


#### Execute Run Loop and Test Pipeline

In [11]:
sample_queries = [
    "Hey, I just checked my credit card statement and I was billed twice ($49.00 each) for my premium subscription this month. Please refund one immediately!",
    "Ever since I updated to version 4.2 last night, the app crashes instantly on launch. I'm on an iPhone 15 Pro. Help, I need this for work today!",
    "I lost my phone over the weekend and I don't have my 2FA backup codes saved anywhere. I'm completely locked out of my account and really need to get back in to check my project files.",
    "Where is the cancel button? I've been digging through my account settings for 20 minutes. I want to cancel my subscription before it auto-renews tomorrow.",
    "Getting consistent 504 Gateway Timeouts when hitting the `/v1/analytics/export` endpoint for payloads greater than 50MB. Is there an active outage?"
]

print("==========================================================")
print("             RUNNING ACTIVE TRIAGE WORKFLOW Pipeline       ")
print("==========================================================")

for idx, query in enumerate(sample_queries, 1):
    print(f"\n⚡ PROCESSING SUPPORT TICKET #{idx}")
    output = triage_pipeline.invoke({"customer_query": query})
    print("\n▼ FINAL CUSTOMER-FACING RESPONSE ▼")
    print(output["final_response"])
    print("=" * 65)
    time.sleep(0.2) 

             RUNNING ACTIVE TRIAGE WORKFLOW Pipeline       

⚡ PROCESSING SUPPORT TICKET #1
[AGENT_TRACE: CLASSIFIER] Intent Match -> billing_dispute | Priority Level -> HIGH
[AGENT_TRACE: RETRIEVAL] Extracted custom documentation matching workflow context.

▼ FINAL CUSTOMER-FACING RESPONSE ▼
# Response to Customer Billing Issue

Thank you for bringing this to our attention, and I sincerely apologize for the inconvenience of the duplicate charge.

I understand how frustrating this must be. I'm taking immediate action to resolve this for you.

**Here's what I'll do:**

1. **Verify the duplicate charge** – I'll pull your transaction records from our payment processor (Stripe) to confirm both $49.00 charges occurred within the same billing cycle for your premium subscription.

2. **Process the refund** – Once verified, I'll immediately initiate a refund for the duplicate charge to your original payment method.

3. **Timeline** – Refunds typically appear back on your credit card within **5